In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00


In [2]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

import pandas as pd

df = dataset["train"].to_pandas()


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [4]:
df_model = df.copy()
df_model = df_model[
    ["instruction", "category", "intent", "response"]
].copy()

df_model = df_model.rename(
    columns={
        "instruction": "customer_message"
    }
)

from sklearn.model_selection import train_test_split

unique_instructions = df_model["customer_message"].unique()

train_instructions, temp_instructions = train_test_split(
    unique_instructions,
    test_size=0.20,
    random_state=42
)

val_instructions, test_instructions = train_test_split(
    temp_instructions,
    test_size=0.50,
    random_state=42
)

train_df = df_model[
    df_model["customer_message"].isin(train_instructions)
].copy()

val_df = df_model[
    df_model["customer_message"].isin(val_instructions)
].copy()

test_df = df_model[
    df_model["customer_message"].isin(test_instructions)
].copy()

from datasets import Dataset

train_intent = train_df[["customer_message", "response"]].copy()
val_intent = val_df[["customer_message", "response"]].copy()
test_intent = test_df[["customer_message", "response"]].copy()



In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_intent,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_intent,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_intent,
    preserve_index=False
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [8]:
def Tokenize_response(row):
  prompt = f"Customer: {row['customer_message']}\n Response:"
  target = row["response"] + tokenizer.eos_token

  prompt_tokens = tokenizer(
        prompt,
        add_special_tokens=False
    )

  full_tokens = tokenizer(
        prompt + target,
        truncation=True,
        max_length=512,
        add_special_tokens=False
    )

  input_ids = full_tokens["input_ids"]
  attention_mask = full_tokens["attention_mask"]

  prompt_length = len(prompt_tokens["input_ids"])

  labels = [-100] * prompt_length + input_ids[prompt_length:]

  return {
      "input_ids": input_ids,
      "attention_mask": attention_mask,
      "labels": labels
  }



In [9]:
train_dataset = train_dataset.map(
    Tokenize_response,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    Tokenize_response,
    remove_columns=val_dataset.column_names
)

test_dataset = test_dataset.map(
    Tokenize_response,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/21482 [00:00<?, ? examples/s]

Map:   0%|          | 0/2663 [00:00<?, ? examples/s]

Map:   0%|          | 0/2727 [00:00<?, ? examples/s]

In [10]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [11]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [13]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [14]:
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_response_15b_qlora",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,

    fp16=True,

    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    optim="paged_adamw_8bit",
)

In [16]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,


    data_collator=data_collator,
)

In [19]:
trainer.train(
    resume_from_checkpoint="/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_15b_qlora/checkpoint-1343"
)

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
2,0.629297,0.648082


TrainOutput(global_step=2686, training_loss=0.31814398946641365, metrics={'train_runtime': 5430.9894, 'train_samples_per_second': 7.911, 'train_steps_per_second': 0.495, 'total_flos': 5.921276083974144e+16, 'train_loss': 0.31814398946641365, 'epoch': 2.0})

In [ ]:
import os
print(os.listdir("./qwen_response_15b_qlora"))

['checkpoint-1343']


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

shutil.copytree(
    "./qwen_response_15b_qlora/checkpoint-1343",
    "/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_15b_qlora/checkpoint-1343",
    dirs_exist_ok=True
)

'/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_15b_qlora/checkpoint-1343'

In [ ]:
import os

print(os.listdir(
    "/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_15b_qlora/checkpoint-1343"
))

['training_args.bin', 'trainer_state.json', 'scaler.pt', 'scheduler.pt', 'tokenizer.json', 'rng_state.pth', 'README.md', 'adapter_config.json', 'optimizer.pt', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_model.safetensors']


In [20]:
import torch

model.eval()
model.to("cuda")

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [21]:
def generate_response(customer_message):
    prompt = f"""Customer: {customer_message}
Response:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated.strip()

In [22]:
customer_message = "I want to cancel my order."
prompt = f"""Customer: {customer_message}
Response:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print("Generated token IDs:", generated_tokens)
print("Decoded generated tokens:", tokenizer.decode(generated_tokens, skip_special_tokens=True))

Generated token IDs: tensor([1654, 3535,  429,  498, 2299, 3330,  311, 9121,  697, 1973],
       device='cuda:0')
Decoded generated tokens: We understand that you're looking to cancel your order


In [23]:
for i in range(10):
    sample = dataset["train"][i]

    customer = sample["instruction"]
    true_response = sample["response"]

    generated_response = generate_response(customer)

    print("=" * 80)
    print("Customer:")
    print(customer)

    print("\nTrue Response:")
    print(true_response)

    print("\nGenerated Response:")
    print(generated_response)

Customer:
question about cancelling order {{Order Number}}

True Response:
I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.

Generated Response:
We understand that you have a question regarding canceling your order with the number #{{Order Number}}. We're here to assist you in resolving any concerns or queries you may have. To better address your question, could you please provide more details? Specifically, we would like to know if there was an issue during the cancellation process or if you need guidance on how to proceed with canceling your order. Your insights will help us improve our services and ensure a seamless experience for all customers.
Customer:
i have a question about cancelling oorder {{Order Number}}

True Response:
I've been informed that you have a question about canceling order {{Order Number}}. I'm here t

In [25]:
test_data = Dataset.from_pandas(
    test_intent,
    preserve_index=False
)

print(test_data)

Dataset({
    features: ['customer_message', 'response'],
    num_rows: 2727
})


In [27]:
test_subset = test_data.shuffle(seed=42).select(range(500))

print(test_subset)

Dataset({
    features: ['customer_message', 'response'],
    num_rows: 500
})


In [28]:
import torch

model.eval()
model.to("cuda")

predictions = []
references = []

for i, sample in enumerate(test_subset):

    customer = sample["customer_message"]
    reference = sample["response"]

    prompt = f"""Customer: {customer}
Response:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    predictions.append(generated)
    references.append(reference)

    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{len(test_data)}")

100/2727
200/2727
300/2727
400/2727
500/2727


KeyboardInterrupt: 

In [29]:
print(len(predictions))
print(len(references))

514
514


In [30]:
!pip install -q evaluate rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00


In [31]:
import evaluate

rouge = evaluate.load("rouge")

rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print(rouge_results)

{'rouge1': np.float64(0.5590846072009628), 'rouge2': np.float64(0.2999796041253769), 'rougeL': np.float64(0.4160816483377835), 'rougeLsum': np.float64(0.44751181377779603)}


In [32]:
bertscore = evaluate.load("bertscore")

bert_results = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en",
    rescale_with_baseline=True
)

print("BERTScore Precision:",
      sum(bert_results["precision"]) / len(bert_results["precision"]))

print("BERTScore Recall:",
      sum(bert_results["recall"]) / len(bert_results["recall"]))

print("BERTScore F1:",
      sum(bert_results["f1"]) / len(bert_results["f1"]))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore Precision: 0.49738869918981415
BERTScore Recall: 0.5127584644591486
BERTScore F1: 0.5047175029604352
